# Scene Occlusion

In [ ]:
import numpy as np
import open3d as o3d
from pathlib import Path
import trimesh
import sys
sys.path.insert(0, '../')
import drm
import copy

DATASET_DIR = Path(r"../../../datasets/V-Scan/data")   # <-- change this
FOLDER_NAME = "bedroom_Leica-P30_1775809921180"
REFERENCE_NAME = "main.txt"
EMPTY_SCENE_NAME = "main_empty.txt"
VARIATION_GLOB = "main_var_*.txt"
VOXEL_SIZE = 0.05
# Distance threshold: the max coverage distance (metres)
THRESHOLD_RESOLUTION = 0.1

%load_ext autoreload
%autoreload 2


In [ ]:
# Load the pointclouds from the folder
dataset_dir = Path(DATASET_DIR)
ref_path = dataset_dir / FOLDER_NAME / REFERENCE_NAME
empty_scene_path = dataset_dir / FOLDER_NAME / EMPTY_SCENE_NAME
var_paths = sorted((dataset_dir/FOLDER_NAME).glob(VARIATION_GLOB))
refPcd,_ = drm.txt_pcd_to_open3d(ref_path, apply_unity_conversion=True)
refPcd = refPcd.voxel_down_sample(VOXEL_SIZE)
emptyPcd,_ = drm.txt_pcd_to_open3d(empty_scene_path, apply_unity_conversion=True)
emptyPcd = emptyPcd.voxel_down_sample(VOXEL_SIZE)
ref_scan_pos = drm.read_transform_matrix(ref_path, apply_unity_conversion=True)[:3,3]

varPcds = []
varPosses = []
for var_path in var_paths:
    varPcd,_ = drm.txt_pcd_to_open3d(var_path, apply_unity_conversion=True)
    varPcd = varPcd.voxel_down_sample(VOXEL_SIZE)
    posTransform = drm.read_transform_matrix(var_path, apply_unity_conversion=True)[:3,3]  # just to check that the transform matrix is read correctly (it is)
    varPcds.append(varPcd)
    varPosses.append(posTransform)

In [ ]:
movedVarPcd = copy.deepcopy(varPcds[0]).translate(-varPosses[0])
newScene = drm.visualise_open3d(movedVarPcd)
newScene.add_geometry(trimesh.creation.axis(origin_size=0.05, axis_length=1.0, axis_radius=0.005))
newScene.show()

In [ ]:
newScene.add_geometry(drm.visualise_open3d(movedVarPcd.get_minimal_oriented_bounding_box(), random_color=True))
newScene.show()

In [ ]:
def build_occlusion_grid(
    reference: o3d.geometry.PointCloud,
    scanner_pos: np.ndarray,
    voxel_size: float = 0.05,
) -> tuple[o3d.geometry.VoxelGrid, o3d.geometry.VoxelGrid]:
    """
    Builds occupied and occluded voxel grids from a reference point cloud.

    Internally shifts the cloud so the scanner is at the origin, rotates into
    the OBB local frame for compact voxelization, then ray marches from the
    scanner through each occupied voxel to find the occluded shadow behind it.
    Both grids are returned in world space.

    Parameters
    ----------
    reference   : point cloud that defines the geometry (e.g. var_pcd)
    scanner_pos : world-space position of the scanner that captured reference
    voxel_size  : edge length of each voxel in metres

    Returns
    -------
    occupied_grid : VoxelGrid — voxels containing at least one point
    occluded_grid : VoxelGrid — empty voxels in the shadow behind geometry
    """
    pts_shifted = np.asarray(reference.points) - scanner_pos

    pts_shifted = np.asarray(reference.points) - scanner_pos

    shifted_pcd        = o3d.geometry.PointCloud()
    shifted_pcd.points = o3d.utility.Vector3dVector(pts_shifted)
    obb    = shifted_pcd.get_minimal_oriented_bounding_box()
    R      = np.asarray(obb.R)
    center = np.asarray(obb.center)

    pts_local = (pts_shifted - center) @ R
    min_bound = pts_local.min(axis=0)
    max_bound = pts_local.max(axis=0)
    grid_size = np.floor((max_bound - min_bound) / voxel_size).astype(int) + 1

    voxel_indices = np.floor((pts_local - min_bound) / voxel_size).astype(int)
    occupied_set  = set(map(tuple, voxel_indices))

    # FIX: split into two steps to avoid precedence bug
    origin_local = (np.zeros(3) - center) @ R
    origin_voxel = (origin_local - min_bound) / voxel_size

    occluded_set = set()
    for voxel in occupied_set:
        ray_dir    = np.array(voxel, dtype=float) + 0.5 - origin_voxel
        ray_length = np.linalg.norm(ray_dir)
        if ray_length == 0:
            continue
        ray_dir_n = ray_dir / ray_length
        t         = ray_length + 1.0
        t_max     = ray_length + np.linalg.norm(grid_size)

        while t < t_max:
            current = np.floor(origin_voxel + t * ray_dir_n).astype(int)
            if np.any(current < 0) or np.any(current >= grid_size):
                break
            current_tuple = tuple(current)
            if current_tuple not in occupied_set:
                occluded_set.add(current_tuple)
            t += 1.0

    def set_to_voxel_grid(voxel_set: set, color: list) -> o3d.geometry.VoxelGrid:
        indices       = np.array(list(voxel_set))
        centres_world = (indices + 0.5) * voxel_size + min_bound
        centres_world = centres_world @ R.T + center + scanner_pos
        pcd           = o3d.geometry.PointCloud()
        pcd.points    = o3d.utility.Vector3dVector(centres_world)
        pcd.paint_uniform_color(color)
        return o3d.geometry.VoxelGrid.create_from_point_cloud(pcd, voxel_size)

    occupied_grid = set_to_voxel_grid(occupied_set, [0.86, 0.24, 0.24])
    occluded_grid = set_to_voxel_grid(occluded_set, [0.24, 0.47, 0.86])

    return occupied_grid, occluded_grid


def get_invisible_points_grid(
    points: o3d.geometry.PointCloud,
    reference: o3d.geometry.PointCloud,
    scanner_pos: np.ndarray,
    voxel_size: float = 0.05,
) -> tuple[o3d.geometry.PointCloud, o3d.geometry.PointCloud]:
    """
    Classifies each point in `points` as invisible (occluded or inside geometry)
    or visible, using the occlusion grid built from `reference`.

    Parameters
    ----------
    points      : point cloud to classify (e.g. uncovered candidate points)
    reference   : point cloud that defines the geometry (e.g. var_pcd)
    scanner_pos : world-space position of the scanner that captured reference
    voxel_size  : must match the value used to build the grid

    Returns
    -------
    invisible : points inside occupied or occluded voxels
    visible   : remaining points
    """
    occupied_grid, occluded_grid = build_occlusion_grid(reference, scanner_pos, voxel_size)

    pts_world      = o3d.utility.Vector3dVector(np.asarray(points.points))
    in_occupied    = np.asarray(occupied_grid.check_if_included(pts_world))
    in_occluded    = np.asarray(occluded_grid.check_if_included(pts_world))
    invisible_mask = in_occupied | in_occluded

    invisible = points.select_by_index(np.where(invisible_mask)[0])
    visible   = points.select_by_index(np.where(~invisible_mask)[0])
    return invisible, visible


def visualise_occlusion_grid(
    occupied_grid: o3d.geometry.VoxelGrid,
    occluded_grid: o3d.geometry.VoxelGrid,
    scanner_pos: np.ndarray,
    voxel_size: float = 0.05,
    show_occupied: bool = True,
    show_occluded: bool = True,
    show_scanner: bool = True,
) -> list[o3d.geometry.Geometry]:
    """
    Returns a list of native o3d geometries ready for show_geometries() or
    o3d.visualization.draw_geometries().

    Parameters
    ----------
    occupied_grid : first return value of build_occlusion_grid
    occluded_grid : second return value of build_occlusion_grid
    scanner_pos   : world-space scanner position, used to place the marker sphere
    voxel_size    : used to size the scanner marker sphere
    """
    geometries = []
    if show_occupied:
        geometries.append(occupied_grid)
    if show_occluded:
        geometries.append(occluded_grid)
    if show_scanner:
        sphere = o3d.geometry.TriangleMesh.create_sphere(radius=voxel_size * 1.5)
        sphere.translate(scanner_pos)
        sphere.paint_uniform_color([1.0, 0.85, 0.0])
        sphere.compute_vertex_normals()
        geometries.append(sphere)
    return geometries

In [ ]:
occupied_grid, occluded_grid = build_occlusion_grid(movedVarPcd, [0,0,0], voxel_size=0.1)

#invisible, visible = get_invisible_points_grid(uncovered_points, var_pcd, scanner_pos)


In [ ]:
geom = visualise_occlusion_grid(occupied_grid, occluded_grid, [0,0,0], voxel_size=0.1)
drm.visualise_open3d(geom).show()

In [ ]:
occupied_grid, occluded_grid = build_occlusion_grid(varPcds[0], varPosses[0], voxel_size=0.1)


In [ ]:
geom = visualise_occlusion_grid(occupied_grid, occluded_grid, varPosses[0], voxel_size=0.1)
drm.visualise_open3d(geom).show()

In [ ]:

# Optionally overlay the point cloud
cloud = drm.o3d_pointcloud_to_trimesh(varPcds[0])
scene.add_geometry(cloud, node_name="cloud")

scene.show()

## Experiments

In [ ]:
import numpy as np
import open3d as o3d
from pathlib import Path
import trimesh
import sys
sys.path.insert(0, '../')
import drm
import drm.combine
import copy

DATASET_DIR = Path(r"../../../datasets/V-Scan/data")   # <-- change this
FOLDER_NAME = "bedroom_Leica-P30_1775809921180"
REFERENCE_NAME = "main.txt"
EMPTY_SCENE_NAME = "main_empty.txt"
VARIATION_GLOB = "main_var_*.txt"
VOXEL_SIZE = 0.05
# Distance threshold: the max coverage distance (metres)
THRESHOLD_RESOLUTION = 0.1

%load_ext autoreload
%autoreload 2


In [ ]:
# Load the pointclouds from the folder
dataset_dir = Path(DATASET_DIR)
ref_path = dataset_dir / FOLDER_NAME / REFERENCE_NAME
empty_scene_path = dataset_dir / FOLDER_NAME / EMPTY_SCENE_NAME
var_paths = sorted((dataset_dir/FOLDER_NAME).glob(VARIATION_GLOB))
refPcd,_ = drm.txt_pcd_to_open3d(ref_path, apply_unity_conversion=True)
refPcd = refPcd.voxel_down_sample(VOXEL_SIZE)
emptyPcd,_ = drm.txt_pcd_to_open3d(empty_scene_path, apply_unity_conversion=True)
emptyPcd = emptyPcd.voxel_down_sample(VOXEL_SIZE)
ref_scan_pos = drm.read_transform_matrix(ref_path, apply_unity_conversion=True)[:3,3]

In [ ]:
occupied_vox, occluded_vox = drm.combine.build_occlusion_grid(refPcd, ref_scan_pos, voxel_size=VOXEL_SIZE*2)
drm.visualise_open3d(drm.combine.visualise_occlusion_grid(occupied_vox, occluded_vox, ref_scan_pos, voxel_size=VOXEL_SIZE*2)).show()

In [ ]:
import numpy as np
import open3d as o3d
from dataclasses import dataclass

# ── Config ────────────────────────────────────────────────────────────────────
ANGLE_MARGIN    = 2    # degrees – azimuth / elevation matching tolerance
# ─────────────────────────────────────────────────────────────────────────────


@dataclass
class PolarPoint:
    r:        float   # radial distance from scanner origin  [metres]
    az:       float   # azimuth  (XY-plane angle from +X)    [degrees]
    el:       float   # elevation (angle from XY-plane)      [degrees]
    xyz:      np.ndarray  # original Cartesian for reference


def cartesian_to_polar(points: np.ndarray, origin: np.ndarray) -> np.ndarray:
    """
    Convert Nx3 Cartesian points to polar (r, azimuth_deg, elevation_deg)
    relative to `origin`.

    Returns Nx3 array: columns = [r, azimuth°, elevation°]
    """
    rel  = points - origin                          # translate to scanner frame
    r    = np.linalg.norm(rel, axis=1)

    az   = np.degrees(np.arctan2(rel[:, 1], rel[:, 0]))          # [-180, 180]
    el   = np.degrees(np.arcsin(np.clip(rel[:, 2] / r, -1, 1)))  # [-90,   90]

    return np.column_stack([r, az, el])


def isolate_new_points(
    refPcd:   o3d.geometry.PointCloud,
    emptyPcd: o3d.geometry.PointCloud,
    threshold: float = VOXEL_SIZE,
) -> tuple[o3d.geometry.PointCloud, np.ndarray]:
    """
    Return points in refPcd whose nearest neighbour in emptyPcd is farther
    than `threshold`.  Uses Open3D's KD-tree for efficiency.

    Returns
    -------
    new_pcd   : PointCloud containing only the isolated points
    new_mask  : boolean array (len = len(refPcd)) – True for kept points
    """
    ref_pts   = np.asarray(refPcd.points)
    empty_kd  = o3d.geometry.KDTreeFlann(emptyPcd)

    distances = np.empty(len(ref_pts))
    for i, pt in enumerate(ref_pts):
        _, _, dist_sq = empty_kd.search_knn_vector_3d(pt, 1)
        distances[i]  = np.sqrt(dist_sq[0])

    new_mask = distances > threshold
    new_pcd  = refPcd.select_by_index(np.where(new_mask)[0])
    return new_pcd, new_mask


def check_voxel_coverage(
    new_pcd:      o3d.geometry.PointCloud,
    voxel_grid:   o3d.geometry.VoxelGrid,
    ref_scan_pos: np.ndarray,
    angle_margin: float = ANGLE_MARGIN,
) -> dict:
    """
    For each voxel in `voxel_grid`, check whether any point in `new_pcd`:
      - is angularly close (az & el within ±angle_margin degrees)
      - has a SMALLER radial distance (i.e. the point is in front of the voxel)

    A voxel is "covered" when at least one such point exists — meaning the
    voxel lies behind a detected object and is occluded from the scanner.

    Returns
    -------
    {
        "covered_voxel_grid"   : o3d.geometry.VoxelGrid,
        "uncovered_voxel_grid" : o3d.geometry.VoxelGrid,
        "covered_pct"          : float,
        "uncovered_pct"        : float,
        "n_covered"            : int,
        "n_uncovered"          : int,
        "n_total"              : int,
    }
    """
    voxels = voxel_grid.get_voxels()
    n_total = len(voxels)

    empty_result = {
        "covered_voxel_grid":   o3d.geometry.VoxelGrid(),
        "uncovered_voxel_grid": o3d.geometry.VoxelGrid(),
        "covered_pct":    0.0,
        "uncovered_pct":  100.0,
        "n_covered":   0,
        "n_uncovered": 0,
        "n_total":     0,
    }

    if n_total == 0 or len(new_pcd.points) == 0:
        return empty_result

    # ── Polar coords of all voxel centres  (Mx3) ───────────────────────────
    voxel_centres = np.array([
        voxel_grid.get_voxel_center_coordinate(v.grid_index)
        for v in voxels
    ])
    polar_vox = cartesian_to_polar(voxel_centres, ref_scan_pos)  # [r, az, el]

    # ── Polar coords of the isolated new points  (Nx3) ─────────────────────
    new_pts   = np.asarray(new_pcd.points)
    polar_pts = cartesian_to_polar(new_pts, ref_scan_pos)        # [r, az, el]

    # ── Vectorised matching: for every voxel find covering points ───────────
    # Shapes: polar_vox (M,3), polar_pts (N,3)
    # Broadcast to (M, N) for angular diff arrays.
    az_diff = np.abs(polar_vox[:, 1, None] - polar_pts[None, :, 1])
    az_diff = np.minimum(az_diff, 360.0 - az_diff)              # wrap-around
    el_diff = np.abs(polar_vox[:, 2, None] - polar_pts[None, :, 2])

    angle_ok = (az_diff <= angle_margin) & (el_diff <= angle_margin)  # (M, N)
    depth_ok = polar_pts[None, :, 0] < polar_vox[:, 0, None]          # point in front of voxel

    covered_mask = np.any(angle_ok & depth_ok, axis=1)                # (M,)

    # ── Split voxel centres into covered / uncovered ────────────────────────
    covered_centres   = voxel_centres[covered_mask]
    uncovered_centres = voxel_centres[~covered_mask]

    def centres_to_voxelgrid(centres: np.ndarray) -> o3d.geometry.VoxelGrid:
        if len(centres) == 0:
            return o3d.geometry.VoxelGrid()
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(centres)
        return o3d.geometry.VoxelGrid.create_from_point_cloud(
            pcd, voxel_size=voxel_grid.voxel_size
        )

    covered_vg   = centres_to_voxelgrid(covered_centres)
    uncovered_vg = centres_to_voxelgrid(uncovered_centres)

    n_covered    = int(covered_mask.sum())
    n_uncovered  = n_total - n_covered
    covered_pct  = round(100.0 * n_covered   / n_total, 2)
    uncovered_pct= round(100.0 * n_uncovered / n_total, 2)

    return {
        "covered_voxel_grid":   covered_vg,
        "uncovered_voxel_grid": uncovered_vg,
        "covered_pct":    covered_pct,
        "uncovered_pct":  uncovered_pct,
        "n_covered":   n_covered,
        "n_uncovered": n_uncovered,
        "n_total":     n_total,
    }


def run_pipeline(
    refPcd:       o3d.geometry.PointCloud,
    emptyPcd:     o3d.geometry.PointCloud,
    ref_scan_pos: np.ndarray,
    voxel_grid:   o3d.geometry.VoxelGrid,
    threshold:    float = VOXEL_SIZE,
    angle_margin: float = ANGLE_MARGIN,
) -> dict:
    """
    End-to-end pipeline:
      1. Isolate points in refPcd not present in emptyPcd
      2. Convert to polar coordinates from ref_scan_pos
      3. Check what fraction are occluded by a voxel behind them

    Returns all intermediate and final results.
    """
    # Step 1 – isolate new points
    new_pcd, new_mask = isolate_new_points(refPcd, emptyPcd, threshold)
    print(f"[1] Isolated {len(new_pcd.points):,} / {len(refPcd.points):,} "
          f"points  (threshold={threshold} m)")

    # Step 2 – polar coordinates
    new_pts   = np.asarray(new_pcd.points)
    polar_pts = cartesian_to_polar(new_pts, ref_scan_pos)
    print(f"[2] Converted to polar  "
          f"r∈[{polar_pts[:,0].min():.2f}, {polar_pts[:,0].max():.2f}] m  "
          f"az∈[{polar_pts[:,1].min():.1f}°, {polar_pts[:,1].max():.1f}°]  "
          f"el∈[{polar_pts[:,2].min():.1f}°, {polar_pts[:,2].max():.1f}°]")

    # Step 3 – voxel coverage
    coverage = check_voxel_coverage(new_pcd, voxel_grid, ref_scan_pos, angle_margin)
    print(f"[3] Voxel coverage  →  "
          f"{coverage['covered_pct']}% covered ({coverage['n_covered']:,} voxels)  |  "
          f"{coverage['uncovered_pct']}% uncovered ({coverage['n_uncovered']:,} voxels)  "
          f"(angle margin=±{angle_margin}°)")

    return {
        "new_pcd":               new_pcd,
        "new_mask":              new_mask,
        "polar_pts":             polar_pts,
        "covered_voxel_grid":    coverage["covered_voxel_grid"],
        "uncovered_voxel_grid":  coverage["uncovered_voxel_grid"],
        "covered_pct":           coverage["covered_pct"],
        "uncovered_pct":         coverage["uncovered_pct"],
        "n_covered":             coverage["n_covered"],
        "n_uncovered":           coverage["n_uncovered"],
        "n_total":               coverage["n_total"],
    }



results = run_pipeline(refPcd, emptyPcd, ref_scan_pos, occluded_vox)
print("\nSummary:")
print(f"  New points      : {len(results['new_pcd'].points):,}")
print(f"  Total voxels    : {results['n_total']:,}")
print(f"  Covered         : {results['covered_pct']}%  ({results['n_covered']:,})")
print(f"  Uncovered       : {results['uncovered_pct']}%  ({results['n_uncovered']:,})")

In [ ]:
drm.visualise_open3d(drm.combine.visualise_occlusion_grid(None, results['uncovered_voxel_grid'], ref_scan_pos, voxel_size=VOXEL_SIZE*2)).show()

In [ ]:
"""
evaluate_dataset.py
───────────────────
Full-dataset occlusion coverage evaluation.

For every scene folder in DATASET_DIR:
  1. Load ref + empty PCD, build occlusion voxel grid
  2. Isolate new points in ref vs empty scene
  3. Check what % of occluded voxels are covered by those points
  4. Aggregate results per scan class, dump summary CSV + console report
"""

from __future__ import annotations

import traceback
from pathlib import Path

import numpy as np
import open3d as o3d
import pandas as pd

import drm
import drm.combine

# ── Config ────────────────────────────────────────────────────────────────────
DATASET_DIR          = Path(r"../../../datasets/V-Scan/data")
FOLDER_GLOB          = "*"
REFERENCE_NAME       = "main.txt"
EMPTY_SCENE_NAME     = "main_empty.txt"

VOXEL_SIZE           = 0.05   # metres
THRESHOLD_RESOLUTION = 0.035    # distance threshold for isolating new points
ANGLE_MARGIN         = 5    # degrees

OUTPUT_CSV           = Path(r"/home/jvermandere/projects/DRM/_output/occlusion_results/occl_results.csv")
# ─────────────────────────────────────────────────────────────────────────────


def get_scan_class(pcd_path: Path) -> str:
    """Extract class by stripping the trailing _<numbers> from the folder name."""
    folder_name = pcd_path.parent.name
    parts = folder_name.rsplit("_", 1)
    if len(parts) == 2 and parts[1].isdigit():
        return parts[0]
    return folder_name


def evaluate_scene(folder: Path) -> dict | None:
    """
    Run the full pipeline for one scene folder.
    Returns a flat result dict, or None if files are missing / an error occurs.
    """
    ref_path   = folder / REFERENCE_NAME
    empty_path = folder / EMPTY_SCENE_NAME

    if not ref_path.exists() or not empty_path.exists():
        print(f"  [SKIP] missing required files")
        return None

    try:
        refPcd, _   = drm.txt_pcd_to_open3d(ref_path,   apply_unity_conversion=True)
        refPcd      = refPcd.voxel_down_sample(VOXEL_SIZE)

        emptyPcd, _ = drm.txt_pcd_to_open3d(empty_path, apply_unity_conversion=True)
        emptyPcd    = emptyPcd.voxel_down_sample(VOXEL_SIZE)

        ref_scan_pos = drm.read_transform_matrix(
            ref_path, apply_unity_conversion=True
        )[:3, 3]

        # Build occlusion grid from the reference scan
        _, occluded_vox = drm.combine.build_occlusion_grid(
            refPcd, ref_scan_pos, voxel_size=VOXEL_SIZE * 2
        )

        # Isolate points in ref that are absent from the empty scene
        new_pcd, _ = isolate_new_points(refPcd, emptyPcd, THRESHOLD_RESOLUTION)

        # Check what fraction of occluded voxels are covered by those points
        cov = check_voxel_coverage(
            new_pcd, occluded_vox, ref_scan_pos, ANGLE_MARGIN
        )

        print(
            f"  occ_voxels={len(occluded_vox.get_voxels()):>6,}  "
            f"new_pts={len(new_pcd.points):>6,}  "
            f"covered={cov['covered_pct']:>6.2f}%  "
            f"uncovered={cov['uncovered_pct']:>6.2f}%"
        )

        return {
            "scene":         folder.name,
            "scan_class":    get_scan_class(ref_path),
            "n_ref_pts":     len(refPcd.points),
            "n_new_points":  len(new_pcd.points),
            "n_total_vox":   cov["n_total"],
            "n_covered":     cov["n_covered"],
            "n_uncovered":   cov["n_uncovered"],
            "covered_pct":   cov["covered_pct"],
            "uncovered_pct": cov["uncovered_pct"],
        }

    except Exception:
        print(f"  [ERROR]")
        traceback.print_exc()
        return None


def run_evaluation() -> pd.DataFrame:
    folders = sorted(f for f in DATASET_DIR.glob(FOLDER_GLOB) if f.is_dir())

    if not folders:
        raise FileNotFoundError(f"No folders found under {DATASET_DIR}")

    rows: list[dict] = []

    for folder in folders:
        print(f"\n{'─'*60}")
        print(f"Scene : {folder.name}")
        result = evaluate_scene(folder)
        if result:
            rows.append(result)

    return pd.DataFrame(rows)


def print_class_summary(df: pd.DataFrame) -> None:
    if df.empty:
        print("\n[!] No results to summarise.")
        return

    numeric = ["n_new_points", "n_total_vox", "n_covered", "n_uncovered",
               "covered_pct", "uncovered_pct"]

    summary = df.groupby("scan_class")[numeric].agg(["mean", "std", "count"]).round(2)

    print(f"\n{'═'*60}")
    print("CLASS SUMMARY  (mean ± std over all scenes in class)")
    print(f"{'═'*60}")

    for cls in summary.index:
        n        = int(summary.loc[cls, ("covered_pct", "count")])
        cov_mean = summary.loc[cls, ("covered_pct", "mean")]
        cov_std  = summary.loc[cls, ("covered_pct", "std")]
        unc_mean = summary.loc[cls, ("uncovered_pct", "mean")]
        unc_std  = summary.loc[cls, ("uncovered_pct", "std")]
        pts_mean = summary.loc[cls, ("n_new_points", "mean")]

        print(
            f"  {cls:<35}  n={n:>3}  "
            f"covered={cov_mean:>6.2f}%±{cov_std:>5.2f}  "
            f"uncovered={unc_mean:>6.2f}%±{unc_std:>5.2f}  "
            f"avg_new_pts={pts_mean:>7.0f}"
        )



df = run_evaluation()

if not df.empty:
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n[✓] Results saved → {OUTPUT_CSV}")

print_class_summary(df)